# 03. ConversationSummaryMemory / ConversationSummaryBufferMemory → 요약 노드 / `SummarizationMiddleware`

| legacy | LangGraph / LangChain v1 |
|---|---|
| `ConversationSummaryMemory(llm=llm)` (대화 전체를 요약문으로 유지) | state 에 `summary` 필드 추가 + 요약 노드 + `RemoveMessage` 로 원문 삭제 |
| `ConversationSummaryBufferMemory(llm=llm, max_token_limit=200)` (최근 원문 + 오래된 부분 요약) | `create_agent(..., middleware=[SummarizationMiddleware(model, trigger=..., keep=...)])` |

> legacy 는 "요약"을 메모리 클래스가 숨겨서 했지만, LangGraph 에서는 **언제 요약할지, 무엇을 지울지**가 그래프 코드에 명시적으로 드러납니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [2]:
from langchain_core.messages import HumanMessage, AIMessage

travel_dialog = [
    ("유럽 여행 패키지의 가격은 얼마인가요?",
     "유럽 14박 15일 패키지의 기본 가격은 3,500유로입니다. 이 가격에는 항공료, 호텔 숙박비, 지정된 관광지 입장료가 포함되어 있습니다."),
    ("여행 중에 방문할 주요 관광지는 어디인가요?",
     "파리의 에펠탑, 로마의 콜로세움, 베를린의 브란덴부르크 문, 취리히의 라인폭포 등 유럽의 유명 관광지를 방문합니다."),
    ("여행자 보험은 포함되어 있나요?",
     "네, 모든 여행자에게 기본 여행자 보험을 제공합니다. 의료비 지원, 긴급 상황 발생 시 지원 등이 포함됩니다."),
    ("항공편 좌석을 비즈니스 클래스로 업그레이드할 수 있나요? 비용은 어떻게 되나요?",
     "가능합니다. 업그레이드 비용은 왕복 기준으로 약 1,200유로 추가됩니다."),
    ("패키지에 포함된 호텔의 등급은 어떻게 되나요?",
     "이 패키지에는 4성급 호텔 숙박이 포함되어 있습니다. 모든 호텔은 중심지에 위치해 관광지 접근성이 좋습니다."),
    ("식사 옵션에 대해 더 자세히 알려주실 수 있나요?",
     "매일 아침 호텔 조식이 제공됩니다. 점심과 저녁은 포함되어 있지 않아 자유롭게 현지 음식을 즐기실 수 있습니다."),
    ("패키지 예약 시 예약금은 얼마인가요? 취소 정책은 어떻게 되나요?",
     "예약 시 500유로의 예약금이 필요합니다. 출발일로부터 30일 전까지는 전액 환불되며, 이후 취소 시 예약금은 환불되지 않습니다. 출발 14일 전부터는 비용의 50%가 청구됩니다."),
]
history = [m for h, a in travel_dialog for m in (HumanMessage(h), AIMessage(a))]
print("메시지 수:", len(history))

메시지 수: 14


## 1. ConversationSummaryMemory 대응: 직접 만드는 요약 그래프
* `State` 에 `summary: str` 를 추가
* 메시지가 6개를 넘으면 `summarize` 노드로 가서 요약을 갱신하고, 최근 2개를 제외한 원문을 `RemoveMessage` 로 삭제

In [3]:
from typing import Literal
from langchain_core.messages import SystemMessage, RemoveMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.checkpoint.memory import InMemorySaver


class State(MessagesState):
    summary: str


def call_model(state: State):
    messages = state["messages"]
    if summary := state.get("summary", ""):
        # legacy 의 history(SystemMessage 요약) 와 같은 위치에 요약을 넣어준다
        messages = [SystemMessage(f"지금까지의 대화 요약:\n{summary}")] + messages
    return {"messages": [llm.invoke(messages)]}


def should_summarize(state: State) -> Literal["summarize", "__end__"]:
    return "summarize" if len(state["messages"]) > 6 else END


def summarize(state: State):
    if summary := state.get("summary", ""):
        instruction = f"기존 요약:\n{summary}\n\n위 대화의 새 내용을 반영해 요약을 갱신하세요. 가격/정책 같은 수치는 빠짐없이 남기세요."
    else:
        instruction = "위 대화를 한국어로 요약하세요. 가격/정책 같은 수치는 빠짐없이 남기세요."
    new_summary = llm.invoke(state["messages"] + [HumanMessage(instruction)]).content
    delete = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]  # 최근 2개만 원문 유지
    return {"summary": new_summary, "messages": delete}


summary_graph = (
    StateGraph(State)
    .add_node("call_model", call_model)
    .add_node("summarize", summarize)
    .add_edge(START, "call_model")
    .add_conditional_edges("call_model", should_summarize)
    .add_edge("summarize", END)
    .compile(checkpointer=InMemorySaver())
)

In [4]:
config = {"configurable": {"thread_id": "travel-1"}}

# 이전 대화 14개 + 새 질문을 넣고 실행 → 답변 후 요약 노드가 실행된다
out = summary_graph.invoke({"messages": history + [HumanMessage("호텔은 몇 성급이라고 하셨죠?")]}, config)
print("답변:", out["messages"][-1].content)

state = summary_graph.get_state(config).values
print("\n남은 원문 메시지 수:", len(state["messages"]))
print("\n[요약] (legacy: memory.load_memory_variables({})['history'])\n", state["summary"])

답변: 패키지에 포함된 호텔은 4성급입니다.

남은 원문 메시지 수: 2

[요약] (legacy: memory.load_memory_variables({})['history'])
 - **패키지 가격**: 3,500유로 (14박 15일)
- **주요 관광지**: 파리의 에펠탑, 로마의 콜로세움, 베를린의 브란덴부르크 문, 취리히의 라인폭포
- **여행자 보험**: 기본 여행자 보험 포함 (의료비 지원 등)
- **항공편 업그레이드**: 비즈니스 클래스로 업그레이드 가능, 비용 약 1,200유로 추가
- **호텔 등급**: 4성급 호텔
- **식사 옵션**: 매일 아침 호텔 조식 제공, 점심과 저녁은 자유롭게 선택
- **예약금**: 500유로
- **취소 정책**: 출발 30일 전까지 전액 환불, 이후 예약금 환불 불가, 출발 14일 전부터는 비용의 50% 청구.


In [5]:
# 원문은 지워졌지만 요약에 남아 있어서 답할 수 있다
out = summary_graph.invoke({"messages": [HumanMessage("예약금은 얼마이고, 식사는 어떤 것이 포함되나요?")]}, config)
print(out["messages"][-1].content)

예약금은 500유로입니다. 식사는 매일 아침 호텔에서 조식이 제공되며, 점심과 저녁은 자유롭게 선택하실 수 있습니다.


## 2. ConversationSummaryBufferMemory 대응: `SummarizationMiddleware`
LangChain v1 에 내장된 미들웨어입니다.

* `trigger=("tokens", N)` : 대화가 N 토큰을 넘으면 요약 실행 (legacy `max_token_limit`)
* `keep=("messages", M)` : 최근 M 개 메시지는 원문 그대로 유지 (나머지는 요약 메시지 1개로 대체)
* `trigger` 는 `("messages", 개수)`, `("fraction", 0.8)`(모델 컨텍스트 대비 비율) 도 가능

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=llm,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=llm,                  # 요약에 사용할 모델 (더 저렴한 모델 지정 가능)
            trigger=("tokens", 200),    # legacy max_token_limit=200 에 해당
            keep=("messages", 4),       # 최근 4개 메시지는 원문 유지
        )
    ],
    checkpointer=InMemorySaver(),
)

cfg = {"configurable": {"thread_id": "travel-2"}}
out = agent.invoke({"messages": history + [HumanMessage("비즈니스 업그레이드 비용이 얼마였죠?")]}, cfg)
print("답변:", out["messages"][-1].content, "\n")

messages = agent.get_state(cfg).values["messages"]
print("state 메시지 수:", len(messages), "(원래 16개)\n")
print("[첫 번째 메시지 = 요약]\n", messages[0].content, "\n")
for m in messages[1:]:
    print(f"{type(m).__name__:14}|", m.content[:80].replace("\n", " "))

답변: 비즈니스 클래스 업그레이드 비용은 왕복 약 1,200유로입니다. 

state 메시지 수: 6 (원래 16개)

[첫 번째 메시지 = 요약]
 Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about a European travel package, seeking details on pricing, itinerary, insurance, upgrades, hotel ratings, and meal options.

## SUMMARY
- The basic price for a 14-night, 15-day European travel package is 3,500 euros, which includes airfare, hotel accommodations, and entrance fees to specified tourist attractions.
- Major tourist sites included in the itinerary are the Eiffel Tower in Paris, the Colosseum in Rome, the Brandenburg Gate in Berlin, and the Rhine Falls in Zurich.
- Basic travel insurance is provided for all travelers, covering medical expenses and support in emergencies.
- Business class upgrades for flights are available at an additional cost of approximately 1,200 euros round trip.
- The package includes accommodations in 4-star hotels, all centrally located for easy access to tourist sites.

## ARTIF

state 의 첫 메시지가 이전 대화 전체를 대신하는 **요약 메시지**로 바뀌고, 그 뒤에 최근 메시지 원문이 남아 있는 것을 확인할 수 있습니다. (legacy `ConversationSummaryBufferMemory` 의 "요약 + 최근 버퍼" 구조와 동일)

* 기본 요약 프롬프트가 영어라 요약도 영어로 만들어집니다. 한국어 요약이 필요하면 `summary_prompt=` 에 `{messages}` 자리표시자를 포함한 한국어 프롬프트를 넘기면 됩니다.
* `keep=("messages", 4)` 라도 AI 의 tool call 과 결과가 떨어지지 않도록 경계를 조정하기 때문에 남는 개수가 조금 다를 수 있습니다.